# CIFAR-10 Image Classification

End-to-end walkthrough: data loading → EDA → preprocessing → model building → training → evaluation → predictions.

This notebook reuses the functions defined in `src/` rather than duplicating logic.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import matplotlib.pyplot as plt

from data_loader import load_raw_data, get_class_names
from preprocess import preprocess_pipeline
from model import build_model
from evaluate import plot_training_history, plot_confusion_matrix
from predict import predict_image, generate_sample_predictions_grid

## 1. Load Data

In [ ]:
(x_train, y_train), (x_test, y_test) = load_raw_data()
class_names = get_class_names()

print('Train shape:', x_train.shape, y_train.shape)
print('Test shape:', x_test.shape, y_test.shape)
print('Classes:', class_names)

## 2. Exploratory Data Analysis

In [ ]:
# Visualize a few sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
idxs = np.random.choice(len(x_train), 10, replace=False)

for ax, idx in zip(axes, idxs):
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx][0]])
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar([class_names[i] for i in unique], counts)
plt.title('Training Set Class Distribution')
plt.xticks(rotation=45)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
data = preprocess_pipeline(x_train, y_train, x_test, y_test, val_size=0.1)

for k, v in data.items():
    print(k, v.shape)

## 4. Build Model

In [ ]:
model = build_model(input_shape=data['x_train'].shape[1:], num_classes=10)
model.summary()

## 5. Train Model

For a full training run, use `src/train.py` (includes checkpointing and history saving). 
Here we run a lightweight version for demonstration purposes.

In [ ]:
history = model.fit(
    data['x_train'], data['y_train'],
    validation_data=(data['x_val'], data['y_val']),
    epochs=10,
    batch_size=64,
    verbose=1
)

## 6. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(data['x_test'], data['y_test'])
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 7. Confusion Matrix

In [ ]:
y_pred_probs = model.predict(data['x_test'])
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(data['y_test'], axis=1)

plot_confusion_matrix(y_true, y_pred, class_names)

## 8. Sample Predictions

In [ ]:
# Note: run this after training + saving the model via src/train.py,
# since this loads the saved model from models/cnn_model.keras
generate_sample_predictions_grid(num_samples=10)

## Conclusion

Summarize final test accuracy, observations about which classes are commonly confused (see confusion matrix), and any next steps (e.g. deeper architecture, transfer learning, more augmentation).